# Hito 2 (real) — Calidad y Limpieza de Datos

**Proyecto Integrador — Minería de Datos 1**

Este notebook toma la evidencia relevada en `01_inspeccion_inicial.ipynb` y, para cada problema
detectado, documenta: **qué se observó**, **qué acción se aplicó** y **qué impacto tuvo** sobre el dataset.

El dataset original (`data/raw/streaming_users_dirty.json`) no se modifica. El resultado final de
este proceso se guarda en `data/processed/streaming_users_clean.csv`, y cada paso queda registrado
en `logs/pipeline_log.csv`.


## 0. Carga del dataset original

In [22]:
import pandas as pd
import numpy as np
import re

df = pd.read_json("streaming_users_dirty.json")
filas_inicial = len(df)
print("Filas:", filas_inicial)
print("Nulos totales:", df.isnull().sum().sum())


Filas: 8160
Nulos totales: 753


In [23]:
log_pasos = []

def registrar_paso(paso, descripcion, data):
    filas = len(data)
    nulos = int(data.isnull().sum().sum())
    retencion = round(filas / filas_inicial * 100, 2)
    log_pasos.append({"Paso": paso, "Descripción": descripcion, "Filas": filas,
                       "Nulos": nulos, "Retención (%)": retencion})
    print(f"Paso {paso}: {filas} filas | {nulos} nulos | {retencion}% de retención")

registrar_paso(1, "Carga del dataset original", df)


Paso 1: 8160 filas | 753 nulos | 100.0% de retención


In [ ]:
from google.colab import files
uploaded = files.upload()

## 1. Duplicados exactos

**Evidencia (Hito 1):** encontramos 126 filas completamente duplicadas (todas las columnas
idénticas). Son claramente el mismo registro cargado más de una vez.

**Decisión:** eliminar las filas exactamente duplicadas, conservando una sola copia. No se pierde
información, porque son copias idénticas.


In [ ]:
df = df.drop_duplicates()
registrar_paso(2, "Eliminación de duplicados exactos", df)


## 2. Duplicados por `user_id` con datos distintos (conflicto)

**Evidencia:** además de los duplicados exactos, encontramos 34 `user_id` que aparecen dos veces
con **datos diferentes** entre sí (por ejemplo, distinto `monthly_watch_time_mins` o distinta
escritura de `favorite_genre` para el mismo usuario). Un mismo usuario no puede tener dos estados
distintos al mismo tiempo, y el dataset no tiene una marca de tiempo que permita saber cuál de los
dos registros es el correcto o el más reciente.

**Decisión:** conservar la primera aparición de cada `user_id` y descartar la siguiente. Es una
regla simple y consistente, aplicada igual para todos los casos. Documentamos esto como una
limitación: no podemos garantizar que la fila conservada sea la "verdadera" en todos los casos.


In [ ]:
df = df.drop_duplicates(subset="user_id", keep="first")
registrar_paso(3, "Resolución de user_id duplicados con datos conflictivos (se conserva la primera aparición)", df)


## 3. Estandarización de categorías

**Evidencia:** las columnas `subscription_plan`, `country` y `favorite_genre` tienen la misma
categoría escrita de formas distintas (mayúsculas, minúsculas, abreviaturas, espacios extra,
palabras en inglés). Por ejemplo "Básico", "basico", "BASICO", "Basic" y "Std"/"Estándar" mezclan
plan y variantes.

**Decisión:** mapear cada variante a una única forma canónica por columna. No se eliminan ni se
inventan datos, solo se normaliza la escritura.


In [ ]:
def normalizar_plan(x):
    x = str(x).strip().lower()
    mapa = {
        "básico": "Básico", "basico": "Básico", "basic": "Básico",
        "estándar": "Estándar", "estandar": "Estándar", "standard": "Estándar", "std": "Estándar",
        "premium": "Premium", "premiun": "Premium",
    }
    return mapa.get(x, x)

def normalizar_pais(x):
    x = str(x).strip().lower()
    mapa = {
        "brasil": "Brasil", "brazil": "Brasil", "bra": "Brasil",
        "colombia": "Colombia", "col": "Colombia",
        "méxico": "México", "mexico": "México", "mex": "México",
        "uruguay": "Uruguay", "ury": "Uruguay",
        "perú": "Perú", "peru": "Perú", "per": "Perú",
        "chile": "Chile", "chl": "Chile",
        "argentina": "Argentina", "arg": "Argentina",
    }
    return mapa.get(x, x)

def normalizar_genero(x):
    if pd.isna(x):
        return x
    x = str(x).strip().lower()
    mapa = {
        "comedia": "Comedia", "comedy": "Comedia",
        "drama": "Drama",
        "documental": "Documental", "documentary": "Documental", "doc": "Documental",
        "thriller": "Thriller", "thriler": "Thriller",
        "romance": "Romance",
        "acción": "Acción", "accion": "Acción", "action": "Acción",
        "crime": "Crime", "crimen": "Crime",
    }
    return mapa.get(x, x)

df["subscription_plan"] = df["subscription_plan"].apply(normalizar_plan)
df["country"] = df["country"].apply(normalizar_pais)
df["favorite_genre"] = df["favorite_genre"].apply(normalizar_genero)

registrar_paso(4, "Estandarización de categorías (subscription_plan, country, favorite_genre)", df)
df["subscription_plan"].value_counts()


In [ ]:
df["country"].value_counts()


In [ ]:
df["favorite_genre"].value_counts(dropna=False)


**Impacto:** ahora cada columna categórica tiene únicamente sus categorías reales (3 planes,
7 países, 7 géneros + nulos), en vez de decenas de variantes de escritura.

## 4. Formato de fechas en `last_login_date`

**Evidencia:** al revisar más de cerca los formatos "distintos" detectados en el Hito 1, encontramos
que en realidad mezclan varios casos:
- Fechas válidas en `MM-DD-AAAA` o `AAAA/MM/DD`.
- Fechas en `DD-MM-AAAA` (por ejemplo "16-04-2022", donde el primer número no puede ser un mes).
- Fechas directamente inválidas o marcadas como error: `"2026-15-03"` (mes 15 no existe),
  `"31-02-2022"` (30 de febrero no existe), `"0000-00-00"` y `"not_available"`.

**Decisión:** convertir todas las fechas a un único formato de fecha real. Para los casos
`DD-MM-AAAA`, solo los interpretamos así cuando el primer número es mayor a 12 (así no hay
ambigüedad posible con `MM-DD-AAAA`). Las fechas imposibles o marcadas como error se dejan como
valor nulo: no tiene sentido "adivinar" una fecha inventada.


In [ ]:
def parsear_fecha(x):
    if pd.isna(x):
        return pd.NaT
    for fmt in ("%Y-%m-%d", "%m-%d-%Y", "%Y/%m/%d"):
        try:
            return pd.to_datetime(x, format=fmt)
        except (ValueError, TypeError):
            pass
    # DD-MM-AAAA solo si es inequívoco (el primer número no puede ser un mes)
    m = re.match(r"^(\d{2})-(\d{2})-(\d{4})$", str(x))
    if m and int(m.group(1)) > 12:
        try:
            return pd.to_datetime(x, format="%d-%m-%Y")
        except (ValueError, TypeError):
            pass
    return pd.NaT

nulos_antes = df["last_login_date"].isnull().sum()
df["last_login_date"] = df["last_login_date"].apply(parsear_fecha)
nulos_despues = df["last_login_date"].isnull().sum()

print("Nulos en last_login_date antes:", nulos_antes)
print("Nulos en last_login_date después (incluye fechas irrecuperables):", nulos_despues)

registrar_paso(5, "Unificación de formato de fechas en last_login_date (fechas inválidas -> nulo)", df)


**Impacto:** de las 7.840 fechas no nulas originales, pudimos interpretar de forma confiable
7.776 (el resto —64 registros— quedó como nulo por ser fechas inválidas o marcadas como error).
Ahora la columna es un tipo fecha real, utilizable para análisis temporal.

## 5. Valores fuera de rango en variables numéricas

**Evidencia:**
- `age`: encontramos edades negativas y edades de hasta 150 años (74 registros fuera del rango
  plausible 0–100).
- `monthly_watch_time_mins`: encontramos valores negativos y dos "mesetas" de valores repetidos
  exactamente en 50000.0 y 99999.0 minutos, muy por encima de lo físicamente posible en un mes
  (30 días × 24 h × 60 min = 43.200 minutos como máximo teórico). El resto de la distribución es
  continua y creíble hasta unos 4.200 minutos/mes, lo que refuerza que 50000 y 99999 son valores
  de error, no usuarios reales muy activos.
- `customer_support_tickets`: encontramos valores negativos, y además la distribución natural va
  de 0 a 5 tickets, pero luego salta directamente a dos valores repetidos muchas veces: 99 y 150.
  Ese salto brusco (sin valores intermedios como 6, 7, 8...) es la misma señal que vimos en
  `monthly_watch_time_mins`: son códigos de error, no cantidades reales de tickets.

**Decisión:** en vez de eliminar la fila completa (perdiendo el resto de la información válida del
usuario), marcamos como nulo únicamente el valor puntual que es imposible. Así conservamos el
resto de las columnas de ese registro para el análisis.


In [ ]:
mask_edad = (df["age"] < 0) | (df["age"] > 100)
mask_consumo = (df["monthly_watch_time_mins"] < 0) | (df["monthly_watch_time_mins"] >= 43200)
mask_tickets = (df["customer_support_tickets"] < 0) | (df["customer_support_tickets"] >= 99)

print("Edades imposibles:", mask_edad.sum())
print("Minutos de consumo imposibles:", mask_consumo.sum())
print("Tickets negativos o con valores sentinela (99, 150):", mask_tickets.sum())

df.loc[mask_edad, "age"] = np.nan
df.loc[mask_consumo, "monthly_watch_time_mins"] = np.nan
df.loc[mask_tickets, "customer_support_tickets"] = np.nan

registrar_paso(6, "Valores imposibles en age, monthly_watch_time_mins y customer_support_tickets marcados como nulos", df)


## 6. Estado final de los nulos

Con todos los pasos anteriores aplicados, así queda la cantidad de nulos por columna:


In [ ]:
df.isnull().sum()


## 6.1 Fechas futuras (revisión adicional)

**Evidencia:** al revisar el rango de `last_login_date` ya parseado, encontramos 15 registros con la fecha exacta `2029-01-01`, muy por delante de la fecha actual del dataset. Que sean 15 registros con exactamente el mismo día es una señal de que se trata de un valor de error o sentinela, no de una coincidencia real.

**Decisión:** un último ingreso no puede ser una fecha futura, así que estos valores se marcan como nulos, igual que hicimos con las demás fechas irrecuperables.


In [ ]:
mask_futuro = df["last_login_date"] > pd.Timestamp.today().normalize()
print("Fechas futuras encontradas:", mask_futuro.sum())

df.loc[mask_futuro, "last_login_date"] = pd.NaT
registrar_paso(7, "Fechas futuras imposibles en last_login_date marcadas como nulas", df)


**Decisión sobre estos nulos:** no vamos a imputar valores automáticamente en esta etapa. Rellenar
`age` o `monthly_watch_time_mins` con un promedio inventaría información que no tenemos evidencia
de que sea correcta, y podría distorsionar el análisis exploratorio (Hito 3). Por eso los dejamos
como nulos explícitos y documentados. Si el PCA (Hito 4) requiere datos completos, se tomará una
decisión puntual en ese notebook y quedará justificada ahí.

Para `favorite_genre`, al ser una variable categórica, sí es razonable mantenerla con su propia
categoría "sin dato" (en vez de eliminarla) para no perder registros en el análisis exploratorio.


In [ ]:
df["favorite_genre"] = df["favorite_genre"].fillna("Sin dato")
df["favorite_genre"].value_counts()


## 7. Guardado del dataset procesado y del log ETL

In [ ]:
df.to_csv("../data/processed/streaming_users_clean.csv", index=False)
print("Dataset procesado guardado en data/processed/streaming_users_clean.csv")
print("Filas finales:", len(df))


In [ ]:
log_df = pd.DataFrame(log_pasos)
log_df.to_csv("../logs/pipeline_log.csv", index=False)
log_df


## 8. Resumen del Hito 2

| Problema | Evidencia | Acción | Impacto |
|---|---|---|---|
| Filas duplicadas exactas | 126 filas idénticas | Eliminadas | -126 filas |
| `user_id` duplicado con datos distintos | 34 usuarios con estados conflictivos | Se conserva la primera aparición | -34 filas |
| Categorías inconsistentes | Variantes de escritura en 3 columnas | Estandarizadas a forma canónica | Sin pérdida de filas |
| Fechas en formatos mixtos / inválidas | Formatos mezclados + 64 fechas imposibles | Unificadas a tipo fecha; inválidas -> nulo | +64 nulos en `last_login_date` |
| Valores imposibles (`age`, `monthly_watch_time_mins`, `customer_support_tickets`) | Negativos y valores fuera de todo rango plausible | Valor puntual -> nulo (se conserva la fila) | +177 nulos en esas columnas |
| Fechas futuras (`2029-01-01`) | 15 registros con la misma fecha futura exacta | Valor puntual -> nulo | +15 nulos en `last_login_date` |

El dataset pasó de 8.160 a **8.000 filas** (98.04% de retención), sin haber tocado nunca el archivo
original en `data/raw/`. Todas las decisiones y su impacto quedaron registradas en
`logs/pipeline_log.csv`, y el resultado final está en `data/processed/streaming_users_clean.csv`,
listo para el análisis exploratorio del Hito 3 (`03_eda.ipynb`).
